# 문항 1 : 뒤죽박죽인 날짜 표기를 하나로 통일하기

### 여러 사이트에서 긁어온 날짜 문자열이 제각각이다. 이를 YYYY-MM-DD 하나로 통일하는 함수를 작성하시오.

```py
samples = [
    "2024.12.24",          "2024-12-24",        "2024/12/24",
    "24.12.24",            "2024년 12월 24일",   "2024년 3월 5일",
    "12/24/2024",          "2024.12.24 14:30",  "등록일 : 2024.12.24",
    "2024-13-45",          "작성일 없음",         "",
]
```


조건

normalize_date(s) -> str | None 함수를 작성할 것

위 12가지를 모두 처리할 것 — 변환 불가능하면 None

두 자리 연도(24.12.24)는 2024로 해석할 것

월·일이 한 자리인 경우(3월 5일)도 03, 05로 채울 것

존재하지 않는 날짜(2024-13-45)는 None으로 처리할 것 — 정규표현식만으론 거를 수 없음. 후처리할 것

12/24/2024(미국식)와 2024/12/24를 구분할 것

각 입력에 대해 어떤 패턴으로 매치됐는지 함께 출력할 것 (기대결과 부분 확인)

※ 매칭된 것을 변수로써 꺼내는 방법 (명명 그룹)

(?P<변수명>정규표현식)
```py
pattern = re.compile(r"(?P<loc>\d{3,4})\-(?P<mid>\d{3,4})\-(?P<last>\d{4})")
m = pattern.search('031-555-3331')
print(m['loc'])
# 031
```


#### 기대 결과
```py
2024.12.24            → 2024-12-24   [ymd_dot]
24.12.24              → 2024-12-24   [ymd_short]
2024년 3월 5일         → 2024-03-05   [ymd_kor]
12/24/2024            → 2024-12-24   [mdy_slash]
2024-13-45            → None         [ymd_dash · 유효하지 않은 날짜]
작성일 없음            → None         [매치 없음]
""                   → None         [빈 값]
```

In [1]:
from datetime import datetime
import re

In [ ]:
PATTERNS = {
    "ymd_dash": re.compile(r"(?P<year>\d{4})-(?P<month>\d{1,2})-(?P<day>\d{1,2})"),
    "ymd_dot": re.compile(r"(?P<year>\d{4})\.(?P<month>\d{1,2})\.(?P<day>\d{1,2})"),
    "ymd_slash": re.compile(r"(?P<year>\d{4})/(?P<month>\d{1,2})/(?P<day>\d{1,2})"),
    "ymd_short": re.compile(r"\b(?P<year>\d{2})\.(?P<month>\d{1,2})\.(?P<day>\d{1,2})\b"),
    "ymd_kor": re.compile(r"(?P<year>\d{4})년\s*(?P<month>\d{1,2})월\s*(?P<day>\d{1,2})일"),
    "mdy_slash": re.compile(r"(?P<month>\d{1,2})/(?P<day>\d{1,2})/(?P<year>\d{4})"),
}

def normalize_date(s: str) -> str | None:       # 날짜를 yyyy-mm-dd로 정리하는 함수 정의
    
    if not s or not s.strip():
        return None

    for name, pattern in PATTERNS.items():
        match = pattern.search(s)
        if match:
            year = match.group("year")
            month = match.group("month")
            day = match.group("day")

            if len(year) ==2:
                year = "20"+year

            month = month.zfill(2)
            day = day.zfill(2)

            try:
                datetime(int(year), int(month), int(day))
                return f"{year}-{month}-{day}"
            except ValueError:
                return None

    return None

In [5]:
samples = [
    "2024.12.24",          "2024-12-24",        "2024/12/24",
    "24.12.24",            "2024년 12월 24일",   "2024년 3월 5일",
    "12/24/2024",          "2024.12.24 14:30",  "등록일 : 2024.12.24",
    "2024-13-45",          "작성일 없음",         "",
]

for sample in samples:                                    # 출력 및 예외 처리
    result = normalize_date(sample)

    pattern_info = "매치 없음"
    if not sample or not sample.strip():
        pattern_info = "빈 값"
    else:
        for name, pattern in PATTERNS.items():
            if pattern.search(sample):
                pattern_info = name
                if result is None:
                    pattern_info += " · 유효하지 않은 날짜"
                break
    print(f"{sample:<20} → {str(result):<12} [{pattern_info}]")        

2024.12.24           → 2024-12-24   [ymd_dot]
2024-12-24           → 2024-12-24   [ymd_dash]
2024/12/24           → 2024-12-24   [ymd_slash]
24.12.24             → 2024-12-24   [ymd_short]
2024년 12월 24일        → 2024-12-24   [ymd_kor]
2024년 3월 5일          → 2024-03-05   [ymd_kor]
12/24/2024           → 2024-12-24   [mdy_slash]
2024.12.24 14:30     → 2024-12-24   [ymd_dot]
등록일 : 2024.12.24     → 2024-12-24   [ymd_dot]
2024-13-45           → None         [ymd_dash · 유효하지 않은 날짜]
작성일 없음               → None         [매치 없음]
                     → None         [빈 값]


# 문항 2 : 서버 액세스 로그 파싱과 집계

### 웹 서버의 액세스 로그를 정규표현식으로 파싱해 분석하시오.

아래 내용을 담은 로그 파일 access.log 를 생성하시오.

203.0.113.42 - - [06/Aug/2026:14:22:31 +0900] "GET /list?page=3 HTTP/1.1" 200 5321 "<https://example.com/>" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
198.51.100.7 - - [06/Aug/2026:14:22:33 +0900] "POST /api/search HTTP/1.1" 429 118 "-" "python-requests/2.31.0"
203.0.113.42 - - [06/Aug/2026:14:22:35 +0900] "GET /detail/9981 HTTP/1.1" 404 209 "<https://example.com/list>" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"

조건

하나의 정규표현식으로 한 줄에서 다음 7개를 추출할 것 
ip / timestamp / method / path / status / bytes / user_agent

명명 그룹 (?P<name>...) 을 사용할 것

형식이 깨진 줄은 건너뛰고, 몇 줄을 건너뛰었는지 출력할 것

다음 세 가지를 집계해 출력할 것

상태코드별 요청 수

4xx·5xx가 발생한 경로 상위 5개

봇으로 의심되는 User-Agent 목록과 그 요청 수 (bot / crawler / spider / python-requests 포함 여부로 판정, 대소문자 무시)

결과를 access_report.csv로 저장할 것


#### 기대결과

파싱 3줄 · 건너뜀 0줄

── 상태코드별 요청 수 ──
status
200    1
404    1
429    1

── 4xx·5xx 발생 경로 상위 5 ──
path
/api/search     1
/detail/9981    1

── 봇 의심 User-Agent ──
user_agent
python-requests/2.31.0    1
  봇 요청 비율 33.3%


In [9]:
from collections import Counter
import csv
import re

In [6]:
log_content = """
203.0.113.42 - - [06/Aug/2026:14:22:31 +0900] "GET /list?page=3 HTTP/1.1" 200 5321 "<https://example.com/>" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
198.51.100.7 - - [06/Aug/2026:14:22:33 +0900] "POST /api/search HTTP/1.1" 429 118 "-" "python-requests/2.31.0"
203.0.113.42 - - [06/Aug/2026:14:22:35 +0900] "GET /detail/9981 HTTP/1.1" 404 209 "<https://example.com/list>" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
"""

with open('access.log', 'w', encoding='utf-8') as f:
    f.write(log_content)

print("파일 생성완료")

파일 생성완료


In [7]:
PATTERNS2 = re.compile(
    r'^(?P<ip>\S+)\s+\S+\s+\S+\s+\[(?P<timestamp>[^\]]+)\]\s+'
    r'"(?P<method>[A-Z]+)\s+(?P<path>\S+)\s+HTTP/[0-9.]+"\s+'
    r'(?P<status>\d{3})\s+(?P<bytes>\d+|-)\s+'
    r'"[^"]*"\s+"(?P<user_agent>[^"]+)"'
)

BOT_PATTERN = re.compile(r"bot|crawler|spider|python-requests", re.IGNORECASE)

print("패턴 생성완료")

패턴 생성완료


In [10]:
parsed_count = 0
skipped_count = 0

status_counter = Counter()
bad_path_counter = Counter()
bot_counter = Counter()

parsed_rows = []
for line in log_content.strip().split('\n'):
    if not line.strip():
        continue

    match = PATTERNS2.search(line)
    if not match:
        skipped_count += 1
        continue
    parsed_count += 1

    data = match.groupdict()
    parsed_rows.append(data)

    status = data["status"]
    status_counter[status] += 1
    if re.match(r"^[45]", status):
        bad_path_counter[data["path"]] += 1
    if BOT_PATTERN.search(data["user_agent"]):
        bot_counter[data["user_agent"]] += 1

print("카운트 완료")

카운트 완료


In [16]:
ff = open("access_report.csv", "w", encoding="utf-8", newline="")
try:
    fields = ["ip", "timestamp", "method", "path", "status", "bytes", "user_agent"]
    writer = csv.DictWriter(ff, fieldnames=fields)
    writer.writeheader()
    writer.writerows(parsed_rows)

    ff.write("\n\n")
    ff.write(f"파싱 {parsed_count}줄 · 건너뜀 {skipped_count}줄\n\n")

    ff.write("── 상태코드별 요청 수 ──\n")
    ff.write('status\n')
    for stat, count in sorted(status_counter.items()):
        ff.write(f"{stat:<7}{count}\n")
    ff.write('\n')

    ff.write("── 4xx·5xx 발생 경로 상위 5 ──\n")
    ff.write("path\n")
    for path, count in bad_path_counter.most_common(5):
        ff.write(f"{path:<16}{count}\n")
    ff.write('\n')

    ff.write("── 봇 의심 User-Agent ──\n")
    ff.write("user_agent\n")
    for ua, count in bot_counter.items():
        ff.write(f"{ua:<26}{count}\n")

    total_bot_requests = sum(bot_counter.values())
    bot_ratio = (total_bot_requests / parsed_count) * 100 if parsed_count > 0 else 0
    ff.write(f"  봇 요청 비율 {bot_ratio:.1f}%")

finally:
    ff.close()